In [4]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"

DISTRICT_OUTPUT = Path("data") / "exposure_district.csv"
RC_OUTPUT = Path("data") / "exposure_rc.csv"

df = pd.read_csv(INPUT_CSV)

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["dtname"] = df["dtname"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# Z-SCORE FUNCTION
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std

# =============================================================================
# CLASSIFICATION
# =============================================================================

def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

# =============================================================================
# DISTRICT MEAN POPULATION
# =============================================================================

district_df = (
    df.groupby(["dtname", "timeperiod"], as_index=False)
      .agg(
          district_population=("sum_population", "mean")
      )
)

# =============================================================================
# MONTHWISE Z-SCORE
# =============================================================================

district_df["population_z"] = (
    district_df.groupby("timeperiod")["district_population"]
    .transform(zscore)
)

# =============================================================================
# EXPOSURE CLASS
# =============================================================================

district_df["exposure"] = district_df["population_z"].apply(classify)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

district_df.to_csv(DISTRICT_OUTPUT, index=False)

# =============================================================================
# APPEND DISTRICT EXPOSURE TO RC DATA
# =============================================================================

rc_df = df.copy()

if "exposure" in rc_df.columns:
    rc_df = rc_df.drop(columns=["exposure"])

rc_df = rc_df.merge(
    district_df[
        [
            "dtname",
            "timeperiod",
            "district_population",
            "population_z",
            "exposure",
        ]
    ],
    on=["dtname", "timeperiod"],
    how="left",
    validate="many_to_one",
)

# =============================================================================
# SAVE RC OUTPUT
# =============================================================================

rc_df.to_csv(RC_OUTPUT, index=False)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"District file saved : {DISTRICT_OUTPUT}")
print(f"RC file saved    : {RC_OUTPUT}")

print("\nDistrict rows:", len(district_df))
print("RC rows:", len(rc_df))

print("\nExposure distribution:")
print(district_df["exposure"].value_counts().sort_index())

print("\nDistrict preview:")
print(
    district_df[
        [
            "dtname",
            "timeperiod",
            "district_population",
            "population_z",
            "exposure",
        ]
    ].head()
)

print("\nRC preview:")
print(
    rc_df[
        [
            "object_id",
            "revenue_ci",
            "dtname",
            "timeperiod",
            "sum_population",
            "district_population",
            "exposure",
        ]
    ].head()
)

District file saved : data/exposure_district.csv
RC file saved    : data/exposure_rc.csv

District rows: 2232
RC rows: 11160

Exposure distribution:
exposure
1     62
2    649
3    901
4    372
5    248
Name: count, dtype: int64

District preview:
  dtname timeperiod  district_population  population_z  exposure
0      0    2021_04          6429.494562     -2.094364         1
1      0    2021_05          6429.494562     -2.094364         1
2      0    2021_06          6429.494562     -2.094364         1
3      0    2021_07          6429.494562     -2.094364         1
4      0    2021_08          6429.494562     -2.094364         1

RC preview:
      object_id       revenue_ci     dtname timeperiod  sum_population  \
0  18-300-00101  Gossaigaon (Pt)  KOKRAJHAR    2021_04   327337.781250   
1  18-300-00102       Bhowraguri  KOKRAJHAR    2021_04    56649.250391   
2  18-300-00103           Dotoma  KOKRAJHAR    2021_04   199774.345312   
3  18-300-00104   Kokrajhar (Pt)  KOKRAJHAR    2021_0